# Daily Challenge — Data Handling and Analysis in Python
## Week 2 — Day 2

**Dataset:** Data Science Job Salaries (`datascience_salaries.csv`)

**Goals**
1. Min-Max normalize the `salary` column → [0, 1].
2. Apply a dimensionality reduction technique (PCA) to the dataset.
3. Group by `experience_level` and compute mean / median salary.

All comments are in English.

In [ ]:
# Standard imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

BASE_DIR = os.getcwd()
print('Working directory:', BASE_DIR)

## 1. Load and explore the dataset

In [ ]:
# The first unnamed column is just a row index from the export -> drop it on the fly
df = pd.read_csv(os.path.join(BASE_DIR, 'datascience_salaries.csv'))
if df.columns[0].startswith('Unnamed') or df.columns[0] == '':
    df = df.drop(columns=df.columns[0])

print('Shape:', df.shape)
df.head()

In [ ]:
df.info()
print('\nMissing values per column:')
print(df.isna().sum())
print('\nUnique experience levels:', df['experience_level'].unique())

## 2. Min-Max Normalization of the `salary` column

Min-Max normalization rescales the values into a bounded `[0, 1]` interval:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

This matters because salaries cover a very wide range — without scaling, distance-based or gradient-based models would be dominated by `salary` and ignore lower-magnitude features.

In [ ]:
# Apply Min-Max normalization using scikit-learn
scaler = MinMaxScaler()
df['salary_normalized'] = scaler.fit_transform(df[['salary']])

df[['salary', 'salary_normalized']].describe().round(4)

In [ ]:
# Visual comparison: original vs normalized salary
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['salary'],            bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('Salary - original ($)')
sns.histplot(df['salary_normalized'], bins=30, kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Salary - Min-Max normalized [0,1]')
plt.tight_layout()
plt.show()

## 3. Dimensionality Reduction with PCA

Most columns of this dataset are **categorical** (`job_title`, `job_type`, `experience_level`, `location`, `salary_currency`). To run PCA we first need a numerical representation, so we:
1. One-hot encode the categorical columns.
2. Standardize the resulting features (PCA is scale-sensitive).
3. Fit PCA and inspect the explained variance.

In [ ]:
# Build a feature matrix: keep salary + one-hot encoded categoricals
cat_cols = ['job_title', 'job_type', 'experience_level', 'location', 'salary_currency']
X = pd.get_dummies(df.drop(columns=['salary_normalized']), columns=cat_cols, drop_first=True)
print('Feature matrix shape before PCA:', X.shape)

In [ ]:
# Standardize every column before PCA (PCA assumes centered + comparable scales)
X_scaled = StandardScaler().fit_transform(X)
print('Scaled feature matrix shape:', X_scaled.shape)

In [ ]:
# Fit a PCA that keeps enough components to explain 95% of the variance
pca_full = PCA(n_components=0.95, svd_solver='full')
X_reduced = pca_full.fit_transform(X_scaled)

print(f'Original number of features : {X_scaled.shape[1]}')
print(f'Reduced number of components: {X_reduced.shape[1]}')
print(f'Total variance explained    : {pca_full.explained_variance_ratio_.sum():.4f}')

In [ ]:
# Plot the cumulative explained variance curve
plt.figure(figsize=(9, 4))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o')
plt.axhline(0.95, color='red', linestyle='--', label='95% threshold')
plt.xlabel('Number of PCA components')
plt.ylabel('Cumulative explained variance')
plt.title('PCA - Cumulative explained variance')
plt.legend()
plt.show()

In [ ]:
# Project the dataset onto the first two principal components for visualization
pca_2d = PCA(n_components=2)
coords = pca_2d.fit_transform(X_scaled)

plt.figure(figsize=(9, 5))
sns.scatterplot(
    x=coords[:, 0], y=coords[:, 1],
    hue=df['experience_level'],
    palette='viridis', alpha=0.7, edgecolor='k'
)
plt.xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% var.)')
plt.ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% var.)')
plt.title('PCA projection of the salary dataset (color = experience level)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Aggregation by `experience_level`

We now compute the **average** and **median** salary per experience level. This directly answers the business question *"how does compensation evolve with experience?"*.

In [ ]:
salary_by_exp = (
    df.groupby('experience_level')['salary']
      .agg(['mean', 'median', 'count'])
      .round(2)
      .sort_values('mean')
)
salary_by_exp

In [ ]:
# Bar chart of mean and median salary per experience level
ax = salary_by_exp[['mean', 'median']].plot(
    kind='bar', figsize=(9, 5), edgecolor='black',
    color=['#4c72b0', '#dd8452']
)
ax.set_title('Average and median salary per experience level')
ax.set_ylabel('Salary ($)')
ax.set_xlabel('Experience level')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot to also show the spread of the salary distribution within each level
plt.figure(figsize=(9, 5))
sns.boxplot(
    data=df,
    x='experience_level',
    y='salary',
    order=salary_by_exp.index.tolist(),
    palette='viridis'
)
plt.title('Salary distribution per experience level')
plt.ylabel('Salary ($)')
plt.xlabel('Experience level')
plt.show()

## 5. Summary of insights

- **Min-Max normalization** mapped salaries from a multi-thousand-dollar range to `[0, 1]`. The *shape* of the distribution is preserved — only the X-axis range changes. This is exactly what we want before feeding the column to KNN, K-Means, SVM or a neural network.
- **PCA** reduced the high-dimensional one-hot-encoded feature space to a much smaller number of components while keeping **95% of the variance**. This both fights the *curse of dimensionality* and lets us visualize the dataset in 2D.
- The **2-D projection** shows that experience levels are *partially* separable in PCA space, which means experience carries signal about the rest of the features (notably `salary` and `job_title`).
- **Aggregation** confirms the expected hierarchy Junior → Mid-level → Senior. Average and median salaries increase monotonically with experience, and the gap between mean and median widens at higher levels — a sign of a **right-skewed** distribution where a few very high paychecks pull the mean up.
- The boxplot adds a final touch: **dispersion grows with seniority**. Senior roles have a much wider salary range than Junior roles, which is consistent with real-world compensation packages (variable bonuses, stock options, location premium).

---
**End of Daily Challenge — Day 2.** Don't forget to push to GitHub.